# Spoken Language Processing 2025-26

# Lab3 - Dialogue Systems

_Bruno Martins_


This lab assignment will introduce tools and concepts related to the development of dialogue systems, exemplifying also the use of automatic speech recognition and text-to-speech models in this particular context. The assignment is also associated with the [FIDAWARD IN Spoken Language Processing award](https://tt.tecnico.ulisboa.pt/en/parcerias-empresariais/aproximacao-ao-talento/premios-de-merito-alunos/english-premio-de-merito-fidaward-in-spoken-language-processing-powered-by-fidelidade/).

Students will be tasked with the development of a turn-based spoken/conversational question answering system, reusing different models available from within the catalogue associated to the HuggingFace Transformers library:

* Speech recognition models (e.g., OpenAI Whisper or SpeechT5).
* Large language models for natural language understanding and generation (e.g., the [Qwen3.5](https://huggingface.co/Qwen/Qwen3.5-0.8B) or [SmolLM2](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) models).
* Text-to-speech models (e.g., SpeechT5).

The first parts of this notebook will guide students in the use of the tools, while the last part presents the main problem that is to be tackled. Note that the first parts also feature **intermediate tasks which students are required to solve**.

To complete the project, student groups must deliver in Fenix an **updated version of this notebook**, featuring the proposed solutions to each task, together with a **small PDF report (2 pages)** outlining the methods that were developed (use the [following Overleaf template](https://www.overleaf.com/latex/templates/interspeech-2026-paper-kit/kzcdqdmkqvbr) for the report). The report can contain a section for each of the parts in the notebook. The set of files corresponding to the solution to Lab3 should be uploaded in Fenix through a .zip file named after the number of the group.

Students are encouraged to modify examples, incorporate different techniques, and in general explore any approach that may permit improving the results. Assessment will be based on task completion, **creativity in the proposed solutions**, and overall accuracy over a benchmark dataset.

### Group identification

Initialize the variable `group_id` with the number that Fenix assigned to your group and `student1_name`, `student1_id`, `student2_name` and `student2_id` with your names and student numbers.

In [3]:
# YOUR CODE HERE
group_id=7
student1_id=106261
student1_name = "Dinis Alves da Silva"
student2_id=106362
student2_name = "Henrique Rodrigues"
print(f"Group number: {group_id}")
print(f"Student 1: {student1_name} ({student1_id})")
print(f"Student 2: {student2_name} ({student2_id})")

Group number: 7
Student 1: Dinis Alves da Silva (106261)
Student 2: Henrique Rodrigues (106362)


In [4]:
assert isinstance(group_id, int) and isinstance(student1_id, int) and isinstance(student2_id, int)
assert isinstance(student1_name, str) and isinstance(student2_name, str)
assert (group_id > 0) and (group_id < 40)
assert (student1_id > 60000) and (student1_id < 120000) and (student2_id > 60000) and (student2_id < 120000)

# Install and import Python packages

NumPy is a Python library that provides functions to process multidimensional arrays. The NumPy documentation is available [here](https://numpy.org/doc/1.24/).

[Librosa](https://librosa.org/) is a Python package for analyzing and processing audio signals. It provides a wide range of tools for tasks such as loading and manipulating audio files, extracting features from audio signals, and visualizing and playing back audio data.

IPython display is a module in the IPython interactive computing environment that provides a set of functions for displaying various types of media in the Jupyter notebook or other IPython-compatible environments. For example, you can use the display() function to display an object in a notebook cell (for example an audio object).

Matplotlib is a popular Python library that allows users to create a wide range of visualizations using a simple and intuitive syntax.

Huggingface transformers provides APIs and tools to easily download and train state-of-the-art pretrained models based on the Transformer architecture. The documentation is available [here](https://huggingface.co/docs/transformers/index) and, for more details, you can check the official [HuggingFace course](https://huggingface.co/course/chapter1/1).

Two libraries associated to HuggingFace transformers, named [datasets](https://huggingface.co/docs/datasets/index) and [evaluate](https://huggingface.co/docs/evaluate/index), respectivly suport the direct access to many well-known datasets and common evaluation metrics used in NLP and speech processing research.

In [5]:
!pip3 install -U transformers
!pip3 install -U fsspec==2025.3.0
!pip3 install -U jiwer
!pip3 install -U librosa
!pip3 install -U datasets
!pip3 install -U evaluate

You should consider upgrading via the 'C:\Users\dinis\OneDrive\Desktop\Processamento_da_fala\processamento_da_fala\Scripts\python.exe -m pip install --upgrade pip' command.


  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.10.0
    Uninstalling fsspec-2025.10.0:
      Successfully uninstalled fsspec-2025.10.0


You should consider upgrading via the 'C:\Users\dinis\OneDrive\Desktop\Processamento_da_fala\processamento_da_fala\Scripts\python.exe -m pip install --upgrade pip' command.
You should consider upgrading via the 'C:\Users\dinis\OneDrive\Desktop\Processamento_da_fala\processamento_da_fala\Scripts\python.exe -m pip install --upgrade pip' command.


  Attempting uninstall: librosa
    Found existing installation: librosa 0.10.1
    Uninstalling librosa-0.10.1:
      Successfully uninstalled librosa-0.10.1


You should consider upgrading via the 'C:\Users\dinis\OneDrive\Desktop\Processamento_da_fala\processamento_da_fala\Scripts\python.exe -m pip install --upgrade pip' command.


You should consider upgrading via the 'C:\Users\dinis\OneDrive\Desktop\Processamento_da_fala\processamento_da_fala\Scripts\python.exe -m pip install --upgrade pip' command.


You should consider upgrading via the 'C:\Users\dinis\OneDrive\Desktop\Processamento_da_fala\processamento_da_fala\Scripts\python.exe -m pip install --upgrade pip' command.


In [6]:
import evaluate
import datasets
import transformers
import numpy as np
import librosa
import librosa.display
from IPython.display import Audio
from matplotlib import pyplot as plt
from transformers import logging

logging.set_verbosity(logging.CRITICAL)

# Using OpenAI Whisper

Whisper is a cutting-edge model for for Automatic Speech Recognition (ASR), developed by OpenAI using a massive dataset of 680,000 hours of multilingual and multitask supervised data collected from the internet, and made available through the HuggingFace Transformers library.

The following example illustrates the use of the Whisper model to transcribe a small audio sample taken from the LibriSpeech dataset (which is available through the HuggingFace datasets library).

More detailed information about Whisper, including information on how to fine-tune the model with task-specific data, is available on a [tutorial in the HuggingFace blog](https://huggingface.co/blog/fine-tune-whisper).

In [7]:
import torch
import librosa
from transformers import logging
from IPython.display import Audio
from transformers import AutoProcessor
from transformers import WhisperForConditionalGeneration, SpeechT5ForSpeechToText
from datasets import load_dataset

# Use the datasets library in streaming mode to avoid loading the entire dataset during initialization
ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation", streaming=True)

processor = AutoProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

# Instead of using "openai/whisper-small", you can alternatively try the "microsoft/speecht5_asr" ASR model
# processor = AutoProcessor.from_pretrained("microsoft/speecht5_asr")
# model = SpeechT5ForSpeechToText.from_pretrained("microsoft/speecht5_asr")

# Access the dataset as an Iterator and retrieve the first element in the dataset
audio = next(iter(ds))["audio"]["array"]

# Resample audio to 16kHz (not needed in the case of the particular dataset used in this example)
# audio = librosa.resample(audio, orig_sr=16000, target_sr=16000)

inputs = processor(audio=audio, sampling_rate=16000, return_tensors="pt")

# You are able to hear the audio
display(Audio(audio, rate=16000))

# You can use the option task="translate" to perform speech translation
generated_ids = model.generate(**inputs)
transcription = processor.batch_decode(generated_ids, max_length=250, skip_special_tokens=True)[0]

print(transcription)

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

ImportError: To support decoding audio data, please install 'torchcodec'.

Automatic Speech Recognition (ASR) models are frequently evaluated through the Word Error Rate ([WER](https://huggingface.co/learn/audio-course/en/chapter5/evaluation#word-error-rate)).

The WER is derived from the Levenshtein distance, working at the word level and aligning the recognized word sequence with the reference (spoken) word sequence using dynamic string alignment. The metric can then be computed as:

WER = (S + D + I) / N = (S + D + I) / (S + D + C),

where S is the number of substitutions, D is the number of deletions, I is the number of insertions, C is the number of correct words, and N is the number of words in the reference (N=S+D+C). The WER value indicates the average number of errors per reference word. The lower the value, the better the performance of the ASR system, with a WER of 0 being a perfect score.

The example below illustrates the computation of the WER for two paired examples of a generated sentence versus a reference sentence. The score produced as output is the average value accross the two examples.

In [ ]:
from evaluate import load

wer = load("wer")
predictions = ["this is the prediction", "there is an other sample"]
references = ["this is the reference", "there is another one"]
wer_score = wer.compute(predictions=predictions, references=references)

print(wer_score)

## Intermediate tasks:

* Collect two audio samples with your own voice, together with an English transcription of the spoken messages. The following [example shows how to record audio from your microphone within a Python notebook running on Google Colab](https://colab.research.google.com/gist/ricardodeazambuja/03ac98c31e87caf284f7b06286ebf7fd/microphone-to-numpy-array-from-your-browser-in-colab.ipynb#scrollTo=H4rxNhsEpr-c), but you can use any other method to collect the audio samples.
* Use the Whisper ASR model to transcribe/translate the two spoken messages that were collected into English text. Notice that Whisper supports speech translation, and hence you can test the model with audio samples involving speech in different languages.
* Use the transcriptions to compute the word error rate.
* Experiment with the use of different recognition models (e.g., larger Whisper models, or more recent ASR models -- check the [Open ASR Leaderboard](https://huggingface.co/spaces/hf-audio/open_asr_leaderboard)) over a larger set of audio/transcription pairs, and see if the error rate changes.

In [ ]:
# Add your solutions to the exercises

# Using LLMs for conditional language generation

[OpenAI GPT-2](https://openai.com/index/gpt-2-1-5b-release/) is a language model based on the Transformer decoder architecture, trained with large scale data collected from the Web using a simple objective: predict the next word, given all of the previous words within some text. The diversity of the dataset causes this simple goal to contain naturally occurring demonstrations of many tasks across diverse domains. Thus, GPT-2 can be used to address problems like question answering, modeling the task as language generation conditioned in the question (plus other relevant additional context).

The following example illustrates the use of the GPT-2 model through the Huggingface Transformers library. The example is written in a generic form, which can easilly be adapted to to be used with other LLMs, including models trained to follow instructions. The example illustrates the use of a particular languade decoding algorithm (i.e., beam search), as well as some of the typicall preprocessing and postprocessing steps, allowing us to directly input any text and getting an intelligible answer.

In [ ]:
from transformers import pipeline, set_seed
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

# make results deterministic
set_seed(42)

# You can also try other models instead of "gpt2", e.g. "Qwen/Qwen3.5-0.8B"
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

messages = [{"role": "user", "content": "Can you tell us what is the capital city of the UK?"}]
if tokenizer.chat_template is None: input_text = messages[0]["content"]
else: input_text = tokenizer.apply_chat_template(messages, enable_thinking=False, add_generation_prompt=True, tokenize=False)

inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=10, num_beams=5)

print("**Sequence handled by the language model**")
print(tokenizer.decode(outputs[0]))
print()
print("**Output**")
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip())


## Intermediate tasks:

* Adapt the example showing how to use GPT-2 to do question answering over the [TriviaQA dataset](http://nlp.cs.washington.edu/triviaqa/) (you can use a [version](https://huggingface.co/datasets/lucadiliello/triviaqa) of this dataset from a previous shared task, which is available from HuggingFace datasets).
* Evaluate the results obtained with different language models. These can include relatively small models trained to follow instructions, for instance from the [SmolLM2](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) or [TinyLlama](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0) families, open versions of larger models such as those from [Qwen](https://huggingface.co/Qwen/Qwen3.5-0.8B) or [DeepSeek](https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B), or models available through APIs such as those supported in the [IAedu platform](https://chat.iaedu.pt/) (in this case using separate Python libraries that support the API calls).
* Evaluate different strategies for improving the use of language models in the question answering task (e.g., considering different prompting strategies, retrieval-augmented generation, models with support for reasoning, parameter efficient fine-tuning, etc.).
* Compute the error over the first 500 examples from the validation split from the TriviaQA dataset, using the [TER metric](https://github.com/huggingface/evaluate/tree/main/metrics/ter) for comparing the generated answers against the ground truth.

Notice that ground-truth answers in TriviaQA correspond to relaively short phrases, e.g. directly answering questions through entity names. In this intermediate step, you should consider designing a strategy that can **generate/extract short answers**.


In [ ]:
# Add your solutions to the exercises

# Using SpeechT5 for converting text-to-speech

Motivated by the success of T5 (Text-To-Text Transfer Transformer) in different natural language processing tasks, the unified-modal SpeechT5 framework explores encoder-decoder pre-training for self-supervised speech/text representation learning.

The model is again conveniently available through the HuggingFace Transformers library. The following example illustrates the use of the SpeechT5 model for generating a spectrogram from a textual input, together with a neural vocoder model for producing a speech signal.

More detailed information about SpeechT5 is available on a [tutorial on the HuggingFace blog](https://huggingface.co/blog/speecht5).

In [ ]:
from transformers import AutoProcessor
from transformers import SpeechT5ForTextToSpeech, SpeechT5HifiGan, set_seed
from IPython.display import Audio
from datasets import load_dataset
import soundfile as sf
import librosa
import torch

# make results deterministic
set_seed(42)

model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")
processor = AutoProcessor.from_pretrained("microsoft/speecht5_tts")

inputs = processor(text="Hello, my dog is cute.", return_tensors="pt")
speaker_embeddings = torch.zeros((1, 512))

# When using SpeechT5 for TTS, you should use "xvector speaker embeddings"
# to customize the output to a particular speaker’s voice characteristics
embeddings_dataset = load_dataset("regisss/cmu-arctic-xvectors", split="validation")
speaker_embeddings = torch.tensor(embeddings_dataset[42]["xvector"]).unsqueeze(0)

spectrogram = model.generate_speech(inputs["input_ids"], speaker_embeddings)
with torch.no_grad(): speech = vocoder(spectrogram)

# You can hear the audio inputs
display(Audio(speech.numpy(), rate=16000))

# You can plot the generated spectrogram
import matplotlib.pyplot as plt
plt.figure()
plt.imshow(spectrogram.T)
plt.show()

# You can plot the generated waveform
librosa.display.waveshow(speech.numpy(), sr=16000)

# You can save the audio to a .wav file
sf.write("tts_example.wav", speech.numpy(), samplerate=16000)

## Intermediate tasks:

* Connect the results from your answer to the previous intermediate task (i.e., conditioned language generation) to the SpeechT5 text-to-speech model, so as to produce speech outputs from the text generated by the model. You can also experiment with the use of other text-to-speech models (e.g., [Bark](https://huggingface.co/suno/bark-small), [CSM](https://huggingface.co/sesame/csm-1b), or [MMS-TTS](https://huggingface.co/facebook/mms-tts-eng)).
* Produce naturally-sounding speech-based answers for the first 5 questions in the validation split from the QA dataset used in the previous exercise.
* Connect also the results from your answer to the first intermediate task (i.e., automated speech recognition) to the SpeechT5 model and the LLM, so as to take spoken questions as input and produce a speech output.
* Take the audio samples from 10 TriviaQA questions (as available in connection to the [SLUE-SQA-5 dataset](https://huggingface.co/datasets/asapp/slue-phase-2), in Huggingface datasets), and evaluate the answers generated for the spoken questions using the TER metric.
* Collect audio samples, with your own voice, for the first 2 questions in the validation split from the TriviaQA dataset, and produce naturally-sounding speech-based answers for these two questions.

Notice that ground-truth answers in TriviaQA correspond to relaively short phrases, e.g. directly answering questions through entity names. Thus, evaluations against this dataset based on TER promote models that also produce short answers. However, for conversational systems, it is often preferable to generate longer answers that sound more natural. Hence, in this intermediate step, you should consider adapting the answer generation strategy in order to **produce naturally-sounding answers**.


In [ ]:
# Add your solutions to the exercises

# Main problem

Students are tasked with joining together the speech recognition/translation, language understanding and generation, and text-to-speech components, in order to build a turn-based conversational spoken question answering approach.

* The method should take as input speech utterances with questions, e.g.loading the audio file(s) that simulate, in an off-line experiment supported by the Python notebook, questions given in multipe turns.
* The language understanding and generation component should use as input a transcription/translation for each speech utterance, and optionally also transcriptions/translations for the previous speech utterances (i.e., the different turns in the same conversation context).
* The language understanding and generation component can explore different strategies for improving answer quality:
  * Use of LLMs trained to follow instructions or capable of performing reasoning, e.g. with reinforcement learning from human feedback.
  * Prompting the language model with retrieved in-context examples.
  * Using parameter-efficient fine-ting with existing conversational question answering datasets (e.g., [the CoQA dataset](https://stanfordnlp.github.io/coqa/), which is [also available](https://huggingface.co/datasets/stanfordnlp/coqa) from HuggingFace datasets).
  * ...
* The text-to-speech component should take as input the results from language generation, and produce a speech output for each question.
* To evaluate the proposed method in an off-line experiment, students must collect small audio samples, with their own voices, for the different questions in one of the instances in the CoQA validation split. Having these files stored in a given directory, the notebook should show the turn-based results produced for the different questions.


Notice that the automated speech recognition, language understanding/generation, and the text-to-speech components, all can explore different approaches from the main suggestions in the previous exercises, although students should attempt to justify their choices (e.g., if changing the automated speech recognition component, show that the alternative achieves a lower WER).

Students can also attempt to further streamline/aggregate the overall approach, e.g. using common models to perform speech recognition, language generation, and text-to-speech (e.g., using models such as [Phi-4-multimodal-instruct](https://huggingface.co/microsoft/Phi-4-multimodal-instruct), which can address the multiple tasks that are involved, or instead using full-duplex speech communication models like [Moshi](https://github.com/kyutai-labs/moshi) or [Moshi-RAG](https://github.com/kyutai-labs/moshi-rag) to would allow one to go beyond turn-based interactions).

In [ ]:
# Add your solutions to the exercises